# Kaggle Full 04 - Backend-Style Inference Sanity (Self-Contained)

This notebook validates the runtime idea directly in Kaggle:
- pretrained detector (`insightface`) for face bbox + landmarks
- liveness model consumes only cropped/resized face


In [ ]:
# !pip install -q insightface onnxruntime-gpu torch opencv-python

In [ ]:
from base64 import b64encode, b64decode
from pathlib import Path
import os

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from insightface.app import FaceAnalysis

In [ ]:
class Detector:
    def __init__(self, det_size=(640, 640)):
        self.app = FaceAnalysis(allowed_modules=['detection'])
        try:
            self.app.prepare(ctx_id=0, det_size=det_size)
        except Exception:
            self.app.prepare(ctx_id=-1, det_size=det_size)

    def detect(self, image_bgr):
        faces = self.app.get(image_bgr, max_num=1)
        if not faces:
            return None
        f = faces[0]
        x1, y1, x2, y2 = [int(v) for v in f.bbox]
        kps = [(float(p[0]), float(p[1])) for p in f.kps.tolist()] if getattr(f, 'kps', None) is not None else []
        return (x1, y1, x2, y2), kps


def clamp_bbox(x1, y1, x2, y2, h, w):
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(x1 + 1, min(x2, w))
    y2 = max(y1 + 1, min(y2, h))
    return x1, y1, x2, y2


def crop_resize(image_bgr, bbox, size=80, margin=0.2):
    x1, y1, x2, y2 = bbox
    mx = int((x2 - x1) * margin)
    my = int((y2 - y1) * margin)
    x1, y1, x2, y2 = clamp_bbox(x1 - mx, y1 - my, x2 + mx, y2 + my, image_bgr.shape[0], image_bgr.shape[1])
    face = image_bgr[y1:y2, x1:x2]
    face = cv2.resize(face, (size, size), interpolation=cv2.INTER_LINEAR)
    return face, (x1, y1, x2, y2)


def load_scripted_model(path):
    if not path.exists():
        return None
    model = torch.jit.load(str(path), map_location='cpu')
    model.eval()
    return model


def score_liveness(model, face_bgr):
    rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    x = rgb.astype(np.float32) / 255.0
    x = np.transpose(x, (2, 0, 1))
    x = torch.from_numpy(x).unsqueeze(0)

    if model is None:
        return 0.5

    with torch.no_grad():
        logits = model(x)
    if logits.ndim == 2 and logits.shape[1] >= 2:
        return float(torch.softmax(logits, dim=1)[0, 1].item())
    if logits.ndim == 2 and logits.shape[1] == 1:
        return float(torch.sigmoid(logits)[0, 0].item())
    return 0.5


def label_from_score(score, t_live=0.9, t_spoof=0.3):
    if score >= t_live:
        return 'live'
    if score <= t_spoof:
        return 'spoof'
    return 'uncertain'


In [ ]:
PREPARED_ROOT = Path('/kaggle/input/datasets/doraemongwa/celeba-spoof-prepared-full/celeba_spoof_prepared_full')
LEGACY_PREP_ROOT = Path('/kaggle/working/celeba_spoof_prepared_full')
SCRIPTED_CANDIDATES = [
    Path('/kaggle/working/celeba_spoof_training_full/best_model_scripted.pt'),
    Path('/kaggle/input/results/celeba_spoof_training_full/best_model_scripted.pt'),
]
SCRIPTED_MODEL = next((p for p in SCRIPTED_CANDIDATES if p.exists()), SCRIPTED_CANDIDATES[0])
TEST_MANIFEST = PREPARED_ROOT / 'manifests' / 'test.csv'

def resolve_manifest_image_path(raw_path):
    text = str(raw_path)
    path = Path(text)
    if path.exists():
        return path

    legacy_prefix = str(LEGACY_PREP_ROOT) + '/'
    if text.startswith(legacy_prefix):
        suffix = text[len(legacy_prefix):]
        candidate = PREPARED_ROOT / suffix
        if candidate.exists():
            return candidate

    marker = 'crops_80x80/'
    if marker in text:
        suffix = text.split(marker, 1)[1]
        candidate = PREPARED_ROOT / 'crops_80x80' / suffix
        if candidate.exists():
            return candidate

    return path

SAMPLE_IMAGE = None
if TEST_MANIFEST.exists():
    test_df = pd.read_csv(TEST_MANIFEST)
    if len(test_df) > 0:
        SAMPLE_IMAGE = resolve_manifest_image_path(test_df.iloc[0].image_path)

if SAMPLE_IMAGE is None:
    SAMPLE_IMAGE = PREPARED_ROOT / 'crops_80x80' / 'test' / '00000000_000000.jpg'

print('scripted path:', SCRIPTED_MODEL)
print('scripted exists:', SCRIPTED_MODEL.exists())
print('test manifest exists:', TEST_MANIFEST.exists())
print('sample path:', SAMPLE_IMAGE)
print('sample exists:', SAMPLE_IMAGE.exists())

In [ ]:
detector = Detector(det_size=(640, 640))
model = load_scripted_model(SCRIPTED_MODEL)

image = cv2.imread(str(SAMPLE_IMAGE))
if image is None:
    raise RuntimeError(f'Could not load sample image: {SAMPLE_IMAGE}')

det = detector.detect(image)
if det is None:
    print('No face detected')
else:
    bbox, kps = det
    face, bbox_expanded = crop_resize(image, bbox, size=80, margin=0.2)
    score = score_liveness(model, face)
    label = label_from_score(score)

    print({'bbox': bbox_expanded, 'landmarks': kps, 'score': score, 'label': label})

    vis = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    x1, y1, x2, y2 = bbox_expanded
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for x, y in kps:
        cv2.circle(vis, (int(x), int(y)), 2, (255, 0, 0), -1)

    plt.figure(figsize=(6, 6))
    plt.imshow(vis)
    plt.axis('off')
    plt.title(f'label={label}, score={score:.3f}')
    plt.show()